# ADS Bibliometric Analysis Pipeline

This notebook is the single entry point for the `ads_bib` package. It executes all steps sequentially, with configuration cells between steps so that downstream parameters can be set based on upstream results.

**Pipeline Steps:**
1. Search & Export — Query ADS API, resolve bibcodes to metadata
2. Translation — Detect languages, translate non-English titles/abstracts
3. Tokenization — Create `full_text`, tokenize with spaCy
4. AND — Optional external author name disambiguation
5. Topic Modeling & Curation — BERTopic + datamapplot + cluster removal
6. Citation Networks — Direct, Co-Citation, Bibliographic Coupling, Author Co-Citation

---
## Setup

In [1]:
import time

from ads_bib.notebook import get_notebook_session

_pipeline_start = time.time()

In [2]:
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)
formatter = logging.Formatter("%(message)s")

if root_logger.handlers:
    for handler in root_logger.handlers:
        handler.setLevel(logging.INFO)
        handler.setFormatter(formatter)
else:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    handler.setFormatter(formatter)
    root_logger.addHandler(handler)

logger = logging.getLogger("pipeline")

---
## Global Run Control

Set run-level switches here. Phase-specific parameters are configured in each phase section below.


In [3]:
# -- RUN CONFIGURATION --
# Set RESET_SESSION=True when you want a fresh run directory.
RESET_SESSION = False
RUN = {
    "run_name": "ADS_Curation_Run",
    "random_seed": 42,
    "openrouter_cost_mode": "hybrid",
}

session = get_notebook_session(
    run_name=RUN["run_name"],
    reset=RESET_SESSION,
    start_time=_pipeline_start,
)
session.set_section("run", RUN)

Run initialized: run_20260312_091644_ADS_Curation_Run
All run outputs will be saved to: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260312_091644_ADS_Curation_Run


---
# Phase 1: Search & Export

Query the NASA ADS API for publications matching a search string, then resolve all bibcodes (publications + references) into full metadata.

### 1.1 Search Configuration

Set query, years, and retrieval limits. These define the corpus scope for all downstream steps.

In [4]:
# --- SEARCH CONFIGURATION ---
# Modify the query string for your research question.
# See: https://ui.adsabs.harvard.edu/help/search/search-syntax

Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"
Set_B = f"citations({Set_A}) AND year:1915-2000"
Set_C = f"citations(citations({Set_A})) AND year:1915-2000"
Set_D = f"({Set_A}) OR ({Set_B}) OR ({Set_C})"
Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

gravity_relativity_terms = [
    '(general AND relativi*)', 'gravit*', '(allgemein* AND relativi*)',
    '"relativité générale"', '"teoria della relatività"',
    '"Gravité quantique"', '"Gravità quantistica"',
    '(einheitlich* AND feld*)', 'Quantengravitation', '"champ unifié"',
    '(unified AND field*)', '"quantum gravity"', 'cosmo*', 'Kosmo*',
]
Set_E = f"abs:({' OR '.join(gravity_relativity_terms)}) AND year:1915-2000"

SEARCH = {
    "query": f"({Set_D}) OR ({Set_T}) OR ({Set_E})",
    "ads_token": None,
    "refresh_search": True,
    "refresh_export": True,
}
# Example quick focus query:
SEARCH["query"] = 'author:"Treder, H*"'
session.set_section("search", SEARCH)

Alternative search ideas:

- Quick focus query: `SEARCH["query"] = 'author:"Treder, H*"'`
- Broader historical build: combine `Set_A`, `Set_B`, `Set_C`, `Set_T`, and a wider anchored `Set_E`.
- Update only `SEARCH["query"]`, then rerun the search/export cells.


### 1.2 Execute Search

Run the ADS search and persist results to the run folder.

In [5]:
session.run_stage("search")

Search


  fetch: 0 [00:00]

  result: 382 publications | 493 unique refs | 788 total refs


### 1.3 Export & Resolve Metadata

Resolve publications and references to structured metadata tables.

In [6]:
session.run_stage("export")


Export


  export:   0%|          | 0/875 [00:00<?, ?it/s]

  publications: 382 | references: 493


In [7]:
if session.publications is not None:
    preview_cols = [
        c for c in ("Bibcode", "Year", "Author", "Title", "References")
        if c in session.publications.columns
    ]
    logger.info(
        "Phase 1 preview: publications=%s rows, columns=%s",
        f"{len(session.publications):,}",
        len(session.publications.columns),
    )
    if preview_cols:
        display(session.publications[preview_cols].head(10))
    else:
        display(session.publications.head(10))

,Bibcode,Year,Author,Title,References
0,1971grun.book.....T,1971,"[Treder, H.J.]",Gravitationstheorie und Äquivalenzprinzip.,[]
1,1967AnP...475..194T,1967,"[Treder, HansJürgen]",Das makroskopische Gravitationsfeld in der ein...,"[1916AnP...354..769E, 1940PhRv...57..147R, 195..."
2,1983AN....304..145T,1983,"[Treder, H.J.]",The problem of the Grand Unification Theory,[1950sts..book.....S]
3,1957AnP...454..369T,1957,"[Treder, H.]",Stromladungsdefinition und elektrische Kraft i...,[1953PhRv...92.1567C]
4,1972drdt.book.....T,1972,"[Treder, H.J.]",Die Relativität der Trägheit.,[]
5,1997GReGr..29..455V,1997,"[von Borzeszkowski, HorstHeino, Treder, HansJü...",The Weyl-Cartan Space Problem in Purely Affine...,"[1950sts..book.....S, 1971GReGr...2..313T, 199..."
6,1975AnP...487..383T,1975,"[Treder, H.J.]",Zur unitarisierten Gravitationstheorie mit lan...,"[1963ForPh..11...81T, 1973AnP...485..229T, 197..."
7,1981AN....302..275T,1981,"[Treder, H. J., Jackisch, G.]",On Soldners Value of Newtonian Deflection of L...,"[1970AnP...480..315T, 1974AnP...486..325T]"
8,1988mqg..book.....V,1988,"[von Borzeszkowski, H.H., Treder, H.J.]",The meaning of quantum gravity.,[]
9,1985FoPh...15..161T,1985,"[Treder, H. J.]",The planckions as largest elementary particles...,[1982FoPh...12.1113V]


---
# Phase 2: Translation

Detect non-English titles and abstracts with fasttext, then translate them using either OpenRouter (API) or a local HuggingFace model.

### 2.1 Translation Configuration

Choose provider, model, and translation target language.

In [8]:
# -- TRANSLATION CONFIGURATION --
# Providers: openrouter, gguf, nllb
TRANSLATE = {
    "enabled": True,
    "provider": "openrouter",
    "model": "google/gemini-3.1-flash-lite-preview",
    "api_key": None,
    "max_workers": 10,
    "max_tokens": 2048,
    "fasttext_model": str(session.paths["models"] / "lid.176.bin"),
}
session.set_section("translate", TRANSLATE)

### 2.2 Language Detection + Translation

Detect language tags and translate non-English titles and abstracts.

In [9]:
session.run_stage("translate")


Translate
  publications non-English: Title=183 | Abstract=35
  references non-English: Title=183 | Abstract=36


  translate:   0%|          | 0/437 [00:00<?, ?it/s]

  cost: $0.0293
  publications: 382 (Title=183 Abstract=35 translated) | references: 493 (Title=183 Abstract=36 translated)


### 2.3 Preview Translated Fields

Inspect translated fields.

In [10]:
if session.publications is not None:
    preview_cols = [
        c for c in ("Title", "Title_en", "Abstract", "Abstract_en")
        if c in session.publications.columns
    ]
    if preview_cols:
        display(session.publications[preview_cols].head(5))

,Title,Title_en,Abstract,Abstract_en
0,Gravitationstheorie und Äquivalenzprinzip.,Theory of gravitation and equivalence principle.,,
1,Das makroskopische Gravitationsfeld in der ein...,The macroscopic gravitational field in unified...,A theory of the gravitation field is proposed ...,A theory of the gravitation field is proposed ...
2,The problem of the Grand Unification Theory,The problem of the Grand Unification Theory,The evolution and fundamental questions of phy...,The evolution and fundamental questions of phy...
3,Stromladungsdefinition und elektrische Kraft i...,Definition of current charge and electric forc...,"Wie Infeld 1950 gezeigt hat, ergibt sich aus d...","As Infeld showed in 1950, the strong system of..."
4,Die Relativität der Trägheit.,The relativity of inertia.,,


---
# Phase 3: Tokenization

Create `full_text` (Title + Abstract) and tokenize publications with spaCy. References are skipped for runtime stability.

In [11]:
import os

# -- TOKENIZATION CONFIGURATION --
TOKENIZE = {
    "enabled": True,
    "spacy_model": "en_core_web_md",
    "batch_size": 512,
    "n_process": min(max((os.cpu_count() or 1) - 1, 1), 8),
    "disable": ("ner", "parser", "textcat"),
    "fallback_model": "en_core_web_md",
    "auto_download": True,
}
session.set_section("tokenize", TOKENIZE)

In [12]:
session.run_stage("tokenize")


Tokenize


  tokenize docs:   0%|          | 0/382 [00:00<?, ?it/s]

  documents: 382 | model: en_core_web_md


In [13]:
if session.refs is not None:
    logger.info(
        "References dataset available at %s",
        session.run.paths["data"] / "references.parquet",
    )

---
# Phase 4: Author Name Disambiguation

Optionally run an external AND package on the curated source datasets. If disabled, the pipeline saves a passthrough checkpoint and continues.

In [14]:
AUTHOR_DISAMBIGUATION = {
    "enabled": False,
    "model_bundle": None,
    "dataset_id": None,
    "force_refresh": False,
    "infer_stage": "full",
}
session.set_section("author_disambiguation", AUTHOR_DISAMBIGUATION)

In [15]:
session.run_stage("author_disambiguation")


Author Disambiguation
  disabled — skipped


---
# Phase 5: Topic Modeling & Curation

Use BERTopic to discover thematic clusters, visualize with datamapplot, then remove unwanted clusters to curate the dataset.

**Important:** Set parameters below based on your dataset size from Phase 1.

### 5.1 Embedding Configuration

Configure embedding provider/model and optional sampling. These settings strongly affect topic quality and runtime/cost.

| Provider | Model Examples | Cost | Notes |
|----------|----------------|------|-------|
| `local` | `google/embeddinggemma-300m`, `BAAI/bge-large-en-v1.5` | None | CPU/GPU, no API needed |
| `huggingface_api` | `huggingface/BAAI/bge-large-en-v1.5` | Per-token | HF Inference API via LiteLLM |
| `openrouter` | `openai/text-embedding-3-large`, `google/gemini-embedding-001` | Per-token | Central billing, thread-pool concurrency |

**Caching**: Embeddings are cached to `data/cache/embeddings/` with SHA-256 fingerprint validation. Changing the dataset or model triggers automatic recomputation.


In [16]:
# -- EMBEDDING CONFIGURATION --
# Providers: openrouter, huggingface_api, local, gguf
TOPIC_MODEL = {
    "sample_size": None,
    "embedding_provider": "openrouter",
    "embedding_model": "qwen/qwen3-embedding-8b",
    "embedding_api_key": None,
    "embedding_batch_size": 64,
    "embedding_max_workers": 20,
    "gguf_embedding_pooling": "cls",
}
session.set_section("topic_model", TOPIC_MODEL)


### 5.2 Compute Embeddings

Create or load cached embeddings for `full_text`.

In [17]:
logger.info(
    "Topic stages reuse shared snapshots plus the embedding and reduction caches."
)

In [18]:
session.run_stage("embeddings")


Embeddings


  embeddings:   0%|          | 0/382 [00:00<?, ?it/s]

  embeddings: 382 x 4,096


### 5.3 Dimensionality Reduction Configuration

Two projections are computed: **5D** (HDBSCAN clustering input) and **2D** (visualization & KDE).

| Method | Strengths | Weaknesses |
|--------|-----------|------------|
| `pacmap` | Fast, good local/global balance | No `densmap` (density-preserving mode) |
| `umap` | Supports `densmap=True` for KDE, better hierarchical structure | Slower, sensitive to `n_neighbors` |

**Key parameters:**

| Parameter | Default | Guidance |
|-----------|---------|----------|
| `n_neighbors` | 80 (PaCMAP) / 80 (UMAP) | Higher = more global structure; lower = more local detail |
| `min_dist` | 0.05 (UMAP only) | Lower = tighter clusters in 2D. Library default 0.1 is too loose for bibliometric data |
| `metric` | `angular`/`cosine` | PaCMAP uses `angular` (auto-converted from `cosine`) |
| `densmap` | `False` (UMAP only) | Set `True` in `PARAMS_2D` if you plan KDE density analysis downstream |


> **Tip**: If you plan KDE analysis on the 2D coordinates, use `method="umap"` with
> `PARAMS_2D = dict(..., densmap=True)`. Without densmap, UMAP inflates dense regions
> in the 2D projection, distorting density estimates.


In [19]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "reduction_method": "pacmap",
    "params_5d": {
        "n_neighbors": 80,
        "metric": "angular",
        "random_state": RUN["random_seed"],
    },
    "params_2d": {
        "n_neighbors": 80,
        "metric": "angular",
        "random_state": RUN["random_seed"],
    },
}
session.set_section("topic_model", TOPIC_MODEL)

In [20]:
session.run_stage("reduction")


Reduction


Reduction (PACMAP):   0%|          | 0/2 [00:00<?, ?it/s]

Note: `n_components != 2` have not been thoroughly tested.
  reduced: 5D 382x5 | 2D 382x2


### 5.4 Clustering Configuration

HDBSCAN discovers topic clusters based on density in the 5D space.

| Parameter | Default | Guidance |
|-----------|---------|----------|
| `MIN_CLUSTER_SIZE` | `max(15, n_docs * 0.001)` | Controls granularity: ~0.1% of docs. Lower = more (smaller) topics |
| `min_samples` | 2–3 | Lower = fewer outliers but noisier clusters. Higher = stricter density |
| `cluster_selection_method` | `"eom"` | Excess of Mass: selects most persistent clusters |
| `cluster_selection_epsilon` | 0.02–0.05 | Absorbs border points; higher = larger clusters, fewer outliers |

**Backend choice:**
- `fast_hdbscan`: Fastest, but no `prediction_data` or `gen_min_span_tree` (no `approximate_predict()`).
- `hdbscan`: Supports `prediction_data=True` for predicting new documents and `gen_min_span_tree=True` for hierarchy analysis.

`BASE_MIN_CLUSTER_SIZE` is only used by Toponymy for micro-cluster granularity.


In [21]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "clustering_method": "fast_hdbscan",
    "cluster_params": {},
    "toponymy_cluster_params": {},
    "toponymy_evoc_cluster_params": {},
    "toponymy_layer_index": 1,
}
session.set_section("topic_model", TOPIC_MODEL)

### 5.5 Backend & LLM Configuration

**Backend matrix:**

| Backend | Clustering Input | LLM Providers | Notes |
|---------|-----------------|---------------|-------|
| `bertopic` | 5D reduced vectors | `local`, `gguf`, `huggingface_api`, `openrouter` | Standard BERTopic + outlier reduction |
| `toponymy` | 5D reduced vectors | `local`, `gguf`, `openrouter` | Hierarchical layers, richer labeling |
| `toponymy_evoc` | Raw embeddings | `local`, `gguf`, `openrouter` | EVoC clusterer, no 5D needed |

**Representation pipeline** (BERTopic):

| Model | Role | Configurable |
|-------|------|--------------|
| `POS` | Part-of-speech filtering (keep nouns, adjectives) | `pos_spacy_model` |
| `KeyBERT` | Semantic keyword re-ranking | Requires local embedding model |
| `MMR` | Maximal Marginal Relevance (diversity) | `mmr_diversity` (0–1) |
| LLM | Final topic label generation | Provider, model, prompt |

Default pipeline: **POS → KeyBERT → MMR → LLM** (sequential). Parallel models stored separately in `topic_aspects_` for comparison.

`MIN_DF` guidance: `max(1, min(5, n_docs // 100))`. Small samples need `min_df=1`; large corpora benefit from higher values to suppress noise terms.


### 5.5a LLM Prompt for Topic Labels

Choose a named prompt via `TOPIC_MODEL["llm_prompt_name"]` or provide a full override via `TOPIC_MODEL["llm_prompt"]`.

Available prompt names:

- `physics`: gravitational physics, astrophysics, cosmology
- `generic`: domain-agnostic scientific labeling


In [22]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "llm_prompt_name": "physics",
    "llm_prompt": None,
}
session.set_section("topic_model", TOPIC_MODEL)

In [23]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "backend": "bertopic",
    "llm_provider": "openrouter",
    "llm_model": "google/gemini-3.1-flash-lite-preview",
    "llm_api_key": None,
    "bertopic_label_max_tokens": 128,
    "toponymy_local_label_max_tokens": 256,
    "pipeline_models": ["POS", "KeyBERT", "MMR"],
    "parallel_models": ["MMR", "POS", "KeyBERT"],
    "toponymy_embedding_model": None,
    "toponymy_max_workers": 10,
    "min_df": None,
    "outlier_threshold": 0.5,
}
session.set_section("topic_model", TOPIC_MODEL)

### 5.5b Fit Topic Model

Run the selected backend (BERTopic or Toponymy) on reduced vectors. This is the most compute/cost-intensive step of the pipeline.

In [24]:
session.run_stage("topic_fit")


Topic Fit
  Topic defaults | docs=382 | min_df=3 | min_cluster_size=15 | base_min_cluster_size=55


  fit:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  outlier refresh:   0%|          | 0/1 [00:00<?, ?it/s]

  backend: bertopic | topics: 5 | outliers: 13


In [25]:
if session.topic_info is not None:
    display(session.topic_info)

,Topic,Count,Name,Representation,MMR,POS,KeyBERT,Representative_Docs
0,-1,13,-1_Foundations of Relativistic and Quantum Cos...,[Foundations of Relativistic and Quantum Cosmo...,"[eddington, einstein, planck, fundamental cons...","[quantum, solar, theory, fundamental, fundamen...","[planckions, cosmology, planck, gravitational ...",[Mach's principle and hidden matter.. Accordin...
1,0,144,0_Foundations of Classical and Relativistic Co...,[Foundations of Classical and Relativistic Cos...,"[cosmology, solar, cosmos, planck, cosmologica...","[physics, cosmology, solar, physical, universe...","[cosmology, cosmological, physics, planckions,...",[Is physics at the threshold of a new stage of...
2,1,86,1_Foundations of General Relativity and Gravit...,[Foundations of General Relativity and Gravita...,"[relativity, general relativity, gravitation, ...","[relativity, gravity, general, general relativ...","[general theory relativity, general relativity...",[Book-Review - Fundamental Principles of Gener...
3,2,72,2_Unified Field Theories and Gravitational Phy...,[Unified Field Theories and Gravitational Phys...,"[einstein, general relativity, field equations...","[field, theory, equations, general, gravitatio...","[general relativity, general relativistic, the...","[On Metric and Matter in Unconnected, Connecte..."
4,3,41,3_Machian Principles and Relativistic Gravitat...,[Machian Principles and Relativistic Gravitati...,"[mach, mach einstein, gravitation, post newton...","[gravitation, post, inertia, principle, mechan...","[theories gravitation, theory gravitation, rel...",[The Post-NEWTONian Effect of Gravitation and ...
5,4,26,4_General Relativistic Covariant Vortex Dynami...,[General Relativistic Covariant Vortex Dynamic...,"[helmholtz, covariant, general relativistic, p...","[theorem, relativistic, covariant, general, po...","[general relativistic, helmholtz, relativistic...",[On the General-Relativistic and Covariant For...


### 5.5c Build Topic DataFrame

Merge topic assignments, 2D coordinates, and embeddings into the main DataFrame.

In [26]:
session.run_stage("topic_dataframe")


Topic Dataframe
  topic dataframe: 382 rows


In [27]:
if session.topic_df is not None:
    display(session.topic_df.head(10))

,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,...,Abstract_en,full_text,tokens,embedding_2d_x,embedding_2d_y,topic_id,Name,MMR,POS,KeyBERT
0,1971grun.book.....T,"[Treder, H.J.]",Gravitationstheorie und Äquivalenzprinzip.,1971,Gravitationstheorie und Äquivalenzprinzip,grun.book,,,,,...,,Theory of gravitation and equivalence principl...,"[theory, gravitation, equivalence, principle]",-1.240812,1.049833,1,1_Foundations of General Relativity and Gravit...,"relativity, general relativity, gravitation, e...","relativity, gravity, general, general relativi...","general theory relativity, general relativity,..."
1,1967AnP...475..194T,"[Treder, HansJürgen]",Das makroskopische Gravitationsfeld in der ein...,1967,Annalen der Physik,AnP,3,475,194,206,...,A theory of the gravitation field is proposed ...,The macroscopic gravitational field in unified...,"[macroscopic, gravitational, field, unified, q...",-1.942069,-0.743557,2,2_Unified Field Theories and Gravitational Phy...,"einstein, general relativity, field equations,...","field, theory, equations, general, gravitation...","general relativity, general relativistic, theo..."
2,1983AN....304..145T,"[Treder, H.J.]",The problem of the Grand Unification Theory,1983,Astronomische Nachrichten,AN,4,304,145,151,...,The evolution and fundamental questions of phy...,The problem of the Grand Unification Theory. T...,"[problem, grand, unification, theory, evolutio...",-1.044213,-0.933582,2,2_Unified Field Theories and Gravitational Phy...,"einstein, general relativity, field equations,...","field, theory, equations, general, gravitation...","general relativity, general relativistic, theo..."
3,1957AnP...454..369T,"[Treder, H.]",Stromladungsdefinition und elektrische Kraft i...,1957,Annalen der Physik,AnP,6,454,369,380,...,"As Infeld showed in 1950, the strong system of...",Definition of current charge and electric forc...,"[definition, current, charge, electric, force,...",-1.422537,-1.020071,2,2_Unified Field Theories and Gravitational Phy...,"einstein, general relativity, field equations,...","field, theory, equations, general, gravitation...","general relativity, general relativistic, theo..."
4,1972drdt.book.....T,"[Treder, H.J.]",Die Relativität der Trägheit.,1972,Die Relativität der Trägheit,drdt.book,,,,,...,,The relativity of inertia..,"[relativity, inertia]",-0.668381,1.150349,1,1_Foundations of General Relativity and Gravit...,"relativity, general relativity, gravitation, e...","relativity, gravity, general, general relativi...","general theory relativity, general relativity,..."
5,1997GReGr..29..455V,"[von Borzeszkowski, HorstHeino, Treder, HansJü...",The Weyl-Cartan Space Problem in Purely Affine...,1997,General Relativity and Gravitation,GReGr,4,29,455,466,...,"According to Poincaré, only the ``epistemologi...",The Weyl-Cartan Space Problem in Purely Affine...,"[weyl, cartan, space, problem, purely, affine,...",-2.002987,-1.014793,2,2_Unified Field Theories and Gravitational Phy...,"einstein, general relativity, field equations,...","field, theory, equations, general, gravitation...","general relativity, general relativistic, theo..."
6,1975AnP...487..383T,"[Treder, H.J.]",Zur unitarisierten Gravitationstheorie mit lan...,1975,Annalen der Physik,AnP,,487,383,400,...,,On Unitarized Gravitation Theory with Long- an...,"[unitarized, gravitation, theory, short, range...",-1.228823,0.298561,1,1_Foundations of General Relativity and Gravit...,"relativity, general relativity, gravitation, e...","relativity, gravity, general, general relativi...","general theory relativity, general relativity,..."
7,1981AN....302..275T,"[Treder, H. J., Jackisch, G.]",On Soldners Value of Newtonian Deflection of L...,1981,Astronomische Nachrichten,AN,6,302,275,,...,,On Soldners Value of Newtonian Deflection of L...,"[soldners, value, newtonian, deflection, light]",-0.125279,1.017025,1,1_Foundations of General Relativity and Gravit...,"relativity, general relativity, g

### 5.6 Visualize Topics

Render an interactive topic map. Use this to inspect cluster semantics before curation.

In [28]:
VISUALIZATION = {
    "enabled": True,
    "title": "ADS Bibliometric Map",
    "subtitle_template": "Topics labeled with {provider}/{model}",
    "dark_mode": True,
}
session.set_section("visualization", VISUALIZATION)

In [29]:
session.run_stage("visualize")


Visualize
  saved: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260312_091644_ADS_Curation_Run\plots\topic_map.html


### 5.7 Curate Dataset

Review the cluster summary, then remove clusters that don't fit your research question.

In [30]:
CURATION = {
    "clusters_to_remove": [],
}
session.set_section("curation", CURATION)

In [31]:
session.run_stage("curate")


Curate
  curated dataset: 382 | topics: 6


### Curated Dataset Artifact

In [32]:
logger.info(
    "Curated dataset artifact: %s",
    session.run.paths["data"] / "publications.parquet",
)

In [33]:
if session.curated_df is not None:
    from ads_bib.curate import get_cluster_summary

    display(get_cluster_summary(session.curated_df))
    display(session.curated_df.head(10))

,topic_id,Count,Percentage,Label
1,0,144,37.7,0_Foundations of Classical and Relativistic Co...
2,1,86,22.5,1_Foundations of General Relativity and Gravit...
3,2,72,18.8,2_Unified Field Theories and Gravitational Phy...
4,3,41,10.7,3_Machian Principles and Relativistic Gravitat...
5,4,26,6.8,4_General Relativistic Covariant Vortex Dynami...
0,-1,13,3.4,-1_Foundations of Relativistic and Quantum Cos...


,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,...,Abstract_en,full_text,tokens,embedding_2d_x,embedding_2d_y,topic_id,Name,MMR,POS,KeyBERT
0,1971grun.book.....T,"[Treder, H.J.]",Gravitationstheorie und Äquivalenzprinzip.,1971,Gravitationstheorie und Äquivalenzprinzip,grun.book,,,,,...,,Theory of gravitation and equivalence principl...,"[theory, gravitation, equivalence, principle]",-1.240812,1.049833,1,1_Foundations of General Relativity and Gravit...,"relativity, general relativity, gravitation, e...","relativity, gravity, general, general relativi...","general theory relativity, general relativity,..."
1,1967AnP...475..194T,"[Treder, HansJürgen]",Das makroskopische Gravitationsfeld in der ein...,1967,Annalen der Physik,AnP,3,475,194,206,...,A theory of the gravitation field is proposed ...,The macroscopic gravitational field in unified...,"[macroscopic, gravitational, field, unified, q...",-1.942069,-0.743557,2,2_Unified Field Theories and Gravitational Phy...,"einstein, general relativity, field equations,...","field, theory, equations, general, gravitation...","general relativity, general relativistic, theo..."
2,1983AN....304..145T,"[Treder, H.J.]",The problem of the Grand Unification Theory,1983,Astronomische Nachrichten,AN,4,304,145,151,...,The evolution and fundamental questions of phy...,The problem of the Grand Unification Theory. T...,"[problem, grand, unification, theory, evolutio...",-1.044213,-0.933582,2,2_Unified Field Theories and Gravitational Phy...,"einstein, general relativity, field equations,...","field, theory, equations, general, gravitation...","general relativity, general relativistic, theo..."
3,1957AnP...454..369T,"[Treder, H.]",Stromladungsdefinition und elektrische Kraft i...,1957,Annalen der Physik,AnP,6,454,369,380,...,"As Infeld showed in 1950, the strong system of...",Definition of current charge and electric forc...,"[definition, current, charge, electric, force,...",-1.422537,-1.020071,2,2_Unified Field Theories and Gravitational Phy...,"einstein, general relativity, field equations,...","field, theory, equations, general, gravitation...","general relativity, general relativistic, theo..."
4,1972drdt.book.....T,"[Treder, H.J.]",Die Relativität der Trägheit.,1972,Die Relativität der Trägheit,drdt.book,,,,,...,,The relativity of inertia..,"[relativity, inertia]",-0.668381,1.150349,1,1_Foundations of General Relativity and Gravit...,"relativity, general relativity, gravitation, e...","relativity, gravity, general, general relativi...","general theory relativity, general relativity,..."
5,1997GReGr..29..455V,"[von Borzeszkowski, HorstHeino, Treder, HansJü...",The Weyl-Cartan Space Problem in Purely Affine...,1997,General Relativity and Gravitation,GReGr,4,29,455,466,...,"According to Poincaré, only the ``epistemologi...",The Weyl-Cartan Space Problem in Purely Affine...,"[weyl, cartan, space, problem, purely, affine,...",-2.002987,-1.014793,2,2_Unified Field Theories and Gravitational Phy...,"einstein, general relativity, field equations,...","field, theory, equations, general, gravitation...","general relativity, general relativistic, theo..."
6,1975AnP...487..383T,"[Treder, H.J.]",Zur unitarisierten Gravitationstheorie mit lan...,1975,Annalen der Physik,AnP,,487,383,400,...,,On Unitarized Gravitation Theory with Long- an...,"[unitarized, gravitation, theory, short, range...",-1.228823,0.298561,1,1_Foundations of General Relativity and Gravit...,"relativity, general relativity, gravitation, e...","relativity, gravity, general, general relativi...","general theory relativity, general relativity,..."
7,1981AN....302..275T,"[Treder, H. J., Jackisch, G.]",On Soldners Value of Newtonian Deflection of L...,1981,Astronomische Nachrichten,AN,6,302,275,,...,,On Soldners Value of Newtonian Deflection of L...,"[soldners, value, newtonian, deflection, light]",-0.125279,1.017025,1,1_Foundations of General Relativity and Gravit...,"relativity, general relativity, g

---
# Phase 6: Citation Networks

Compute citation networks from the curated dataset and export to GEXF (Gephi), Graphology JSON (Sigma.js), CSV, and/or Web of Science format.

### 6.1 Citation Configuration

Set network metrics, thresholds, filters, and export formats.

In [34]:
CITATIONS = {
    "metrics": [
        "direct",
        "co_citation",
        "bibliographic_coupling",
        "author_co_citation",
    ],
    "min_counts": {
        "direct": 5,
        "co_citation": 20,
        "bibliographic_coupling": 20,
        "author_co_citation": 10,
    },
    "authors_filter": None,
    "output_format": "gexf",
}
session.set_section("citations", CITATIONS)

### 6.2 Build Node Table & Compute Networks

Build node metadata and compute selected citation networks.

In [35]:
session.run_stage("citations")


Citations
  author_co_citation: 59 edges | direct: 120 edges


### 6.3 Export Web of Science Format

Export the curated corpus in WOS text format for CiteSpace/VOSviewer.

In [36]:
suffix = "_filtered" if session.config.citations.authors_filter else ""
logger.info(
    "WOS export artifact: %s",
    session.run.paths["data"] / f"download_wos_export{suffix}.txt",
)

---
# Summary

Final dataset statistics and output file listing.

In [37]:
logger.info("Run artifacts: %s", session.run.paths["root"])

In [38]:
session.save_summary()

PIPELINE COMPLETE
Publications:     382
References:       493
Curated dataset:  382
Topics found:     6

Output files:
  config_used.yaml (0.0 MB)
  data\author_co_citation.gexf (0.0 MB)
  data\direct.gexf (0.4 MB)
  data\download_wos_export.txt (0.2 MB)
  data\publications.parquet (0.3 MB)
  data\publications_disambiguated.parquet (0.3 MB)
  data\publications_tokenized.json (0.7 MB)
  data\publications_translated.json (0.4 MB)
  data\references.parquet (0.3 MB)
  data\references_disambiguated.parquet (0.3 MB)
  data\references_translated.json (0.6 MB)
  logs\runtime.log (0.0 MB)
  plots\topic_map.html (0.2 MB)

CostTracker Summary (compact)
  translation | google/gemini-3.1-flash-lite-preview | tokens(total=50,409, prompt=37,017, completion=13,392) | calls=2 | cost=$0.0293
  embeddings | qwen/qwen3-embedding-8b | tokens(total=26,013, prompt=26,013, completion=0) | calls=1 | cost=n/a
  llm_labeling | google/gemini-3.1-flash-lite-preview | tokens(total=3,098, prompt=3,039, completion=59